# 📘 Notebook 09: Deep Q-Networks

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module C: Deep Value-Based Learning · Notebook 2 of 4 in this module · (9 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain **experience replay** and implement a replay buffer
- Explain the **target network** and implement its slow update
- Write the DQN loss for a **minibatch** of transitions in PyTorch
- Assemble the complete **DQN** algorithm and show that it removes the instability of Notebook 08
- Demonstrate by **ablation** that the target network and the replay buffer are both needed
- Train an agent to land a spacecraft in **Lunar Lander**, and save the trained network

> **Prerequisites:** Notebook 08 (a network that outputs one value per action, the Q-learning loss, and the two causes of instability) and Notebook 05 (off-policy learning).
>
> 🔭 **Why this notebook matters later:** DQN is the algorithm that started deep reinforcement learning as a field. The replay buffer and the target network built here return in Notebooks 10, 11 and 15, and the training loop is the template for every agent that follows.

> 📦 **Libraries used in this notebook.**
>
> - **PyTorch** for the networks.
> - **Gymnasium** with its `box2d` extra, which contains the Lunar Lander environment. Box2D is a physics engine and needs the build tool `swig`, so the installation takes a little longer than usual.
>
> No GPU is needed. The CartPole experiments take a few seconds each. Training the lander takes a few minutes, and the notebook tells you when that cell comes.

In [ ]:
%pip install -q swig
%pip install -q "gymnasium[classic-control,box2d]>=1.3" imageio imageio-ffmpeg torch

In [ ]:
import base64
import copy
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=30, width=420):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. Two problems, two cures

Notebook 08 replaced the Q-table by a neural network and found that learning became erratic, for two reasons:

| Problem | Cure | Section |
|---|---|---|
| The agent learns from each step as it happens. Consecutive steps are alike, and what was learned earlier is overwritten. | **Experience replay** | 2 |
| The target $R + \gamma \max_a \hat{q}(S', a)$ is computed by the network being trained, so it moves with every update. | **Target network** | 3 |

A group at DeepMind led by Volodymyr Mnih combined the two with a convolutional network and showed, in papers of 2013 and 2015, that a single agent could learn to play many Atari video games from the screen image. They called it the **Deep Q-Network (DQN)**.

## 2. Experience replay

### Intuition first
A student who only ever looks at the exercise in front of them, and throws it away afterwards, forgets Monday's material by Friday. A better student keeps all the past exercises in a folder and, every evening, redoes a handful picked at random. The topics come up mixed together, and nothing is forgotten merely because it was not seen for a few days.

### Formal definition
Every step the agent takes is a **transition** $(S, A, R, S', \text{terminated})$. A **replay buffer** stores the most recent transitions, up to a fixed capacity. To learn, the agent draws a random **minibatch** from the buffer and updates the network on it.

### Why it matters
- **The samples are mixed.** A minibatch contains steps from many episodes and many regions of the state space, so no single region dominates an update. This is the shuffling that supervised learning relies on.
- **Experience is reused.** A transition is used in many updates before it leaves the buffer. Interaction with the environment is usually the expensive part, so this matters.

Replay is only legitimate for an **off-policy** method. The transitions in the buffer were produced by older versions of the policy. Q-learning can learn from them because its target does not depend on which policy generated the data (Notebook 05, Section 5).

In [ ]:
class ReplayBuffer:
    """A fixed-size memory of past steps. When it is full, the oldest step is overwritten."""

    def __init__(self, capacity, state_size):
        self.states = np.zeros((capacity, state_size), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_size), dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.capacity, self.size, self.position = capacity, 0, 0

    def add(self, state, action, reward, next_state, terminated):
        i = self.position
        self.states[i], self.actions[i], self.rewards[i] = state, action, reward
        self.next_states[i], self.terminated[i] = next_state, float(terminated)
        self.position = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, rng):
        index = rng.integers(0, self.size, batch_size)
        return (torch.as_tensor(self.states[index]), torch.as_tensor(self.actions[index]),
                torch.as_tensor(self.rewards[index]), torch.as_tensor(self.next_states[index]),
                torch.as_tensor(self.terminated[index]))


rng = np.random.default_rng(0)
buffer = ReplayBuffer(capacity=1000, state_size=4)

env = gym.make("CartPole-v1")
state, info = env.reset(seed=0)
for step in range(300):
    action = env.action_space.sample()
    next_state, reward, terminated, truncated, info = env.step(action)
    buffer.add(state, action, reward, next_state, terminated)
    state = next_state
    if terminated or truncated:
        state, info = env.reset()

states, actions, rewards, next_states, terminated = buffer.sample(5, rng)
print("stored transitions:", buffer.size)
print("a minibatch of 5:  states", tuple(states.shape), " actions", actions.tolist(), " terminated", terminated.tolist())

## 3. The target network

### Intuition first
Try to hit a target that jumps a little every time you take aim, in a direction that depends on where you aimed. You would do much better if somebody held the target still for a while, let you improve, and only then moved it to a better place.

### Formal definition
DQN keeps **two** networks with the same architecture:

- the **online network**, with weights $\mathbf{w}$, chooses the actions and is trained by gradient descent;
- the **target network**, with weights $\mathbf{w}^-$, is used only to compute the targets:

$$\text{target} = R + \gamma \max_a \hat{q}(S', a; \mathbf{w}^-)$$

The target network is never trained directly. It follows the online network slowly. The original DQN copied the weights every few thousand steps. We use the smoother variant that is now more common, a **soft update** after every training step:

$$\mathbf{w}^- \leftarrow (1 - \tau)\,\mathbf{w}^- + \tau\,\mathbf{w}$$

With $\tau = 0.005$ the target network moves half a percent of the way towards the online network each time. The targets still improve, but they change slowly enough for the online network to make progress against them.

## 4. The loss for a minibatch

Notebook 08 computed the loss for one transition. A minibatch is processed in one go, and two lines of PyTorch deserve an explanation.

- `online(states)` has one row per transition and one column per action. We need, in each row, the value of the action that was **actually taken**. `gather(1, actions.unsqueeze(1))` picks exactly those entries.
- The factor `(1 - terminated)` switches off the future term for transitions that ended the game, the rule we have followed since Notebook 05.

In [ ]:
def make_network(state_size, n_actions, hidden=128):
    return nn.Sequential(nn.Linear(state_size, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, n_actions))


def dqn_loss(online, target, batch, gamma):
    states, actions, rewards, next_states, terminated = batch
    with torch.no_grad():                                               # targets are fixed numbers
        best_next = target(next_states).max(dim=1).values               # from the TARGET network
        targets = rewards + gamma * (1 - terminated) * best_next
    predicted = online(states).gather(1, actions.unsqueeze(1)).squeeze(1)   # value of the action taken
    return nn.functional.smooth_l1_loss(predicted, targets)


torch.manual_seed(0)
online = make_network(4, 2)
target = copy.deepcopy(online)
print(f"loss on a random minibatch: {dqn_loss(online, target, buffer.sample(64, rng), gamma=0.99).item():.3f}")

The loss function is `smooth_l1_loss`, also called the **Huber loss**. It behaves like the squared error for small errors and like the absolute error for large ones, so a single wildly wrong target cannot produce an enormous gradient.

## 5. The complete algorithm

Everything is now in place:

1. Act epsilon-greedily with the online network and **store** the transition.
2. Every few steps, **sample** a minibatch and take one gradient step on the loss.
3. **Soft-update** the target network.

In [ ]:
def dqn(env, steps, hidden=128, learning_rate=5e-4, gamma=0.99, batch_size=64, buffer_size=100000,
        warm_up=1000, train_every=4, tau=0.005, explore_steps=20000, epsilon_end=0.02,
        target_score=None, seed=0, log=None):
    """Deep Q-Network. Returns the trained online network and the return of every episode."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    state_size, n_actions = env.observation_space.shape[0], env.action_space.n
    online = make_network(state_size, n_actions, hidden)
    target = copy.deepcopy(online)
    optimiser = torch.optim.Adam(online.parameters(), lr=learning_rate)
    buffer = ReplayBuffer(buffer_size, state_size)
    returns, episode_return = [], 0.0
    state, info = env.reset(seed=seed)

    for step in range(1, steps + 1):
        # 1. act and store
        epsilon = max(epsilon_end, 1 - step / explore_steps)
        if rng.random() < epsilon:
            action = int(rng.integers(n_actions))
        else:
            with torch.no_grad():
                action = int(online(torch.as_tensor(state, dtype=torch.float32)).argmax())
        next_state, reward, terminated, truncated, info = env.step(action)
        buffer.add(state, action, reward, next_state, terminated)
        state, episode_return = next_state, episode_return + reward

        if terminated or truncated:
            returns.append(episode_return)
            episode_return = 0.0
            state, info = env.reset()
            if log and len(returns) % log == 0:
                print(f"step {step:>7,}   episode {len(returns):>4}   "
                      f"average return of the last {log} episodes: {np.mean(returns[-log:]):7.1f}")
            if target_score is not None and len(returns) >= 100 and np.mean(returns[-100:]) >= target_score:
                break                                              # good enough: stop early

        # 2. learn from a random minibatch
        if buffer.size >= warm_up and step % train_every == 0:
            loss = dqn_loss(online, target, buffer.sample(batch_size, rng), gamma)
            optimiser.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(online.parameters(), 10.0)    # limit the size of one update
            optimiser.step()
            # 3. let the target network follow slowly
            with torch.no_grad():
                for w, w_target in zip(online.parameters(), target.parameters()):
                    w_target.mul_(1 - tau).add_(tau * w)
    return online, np.array(returns)


def evaluate(env, network, episodes=30):
    """Average return of the greedy policy."""
    total = 0.0
    for episode in range(episodes):
        state, info = env.reset(seed=90000 + episode)
        terminated = truncated = False
        while not (terminated or truncated):
            with torch.no_grad():
                action = int(network(torch.as_tensor(state, dtype=torch.float32)).argmax())
            state, reward, terminated, truncated, info = env.step(action)
            total += reward
    return total / episodes

Two practical details in the code: learning starts only after a **warm-up** of 1,000 random steps, so that the first minibatches are not all alike, and one gradient step is taken every 4 environment steps, which is a common balance between acting and learning.

## 6. CartPole again

The same experiment as in Notebook 08: three seeds, 20,000 steps each.

In [ ]:
CARTPOLE = dict(learning_rate=2.5e-4, explore_steps=8000)

runs = [dqn(env, 20000, seed=seed, **CARTPOLE) for seed in range(3)]


def smooth(values, window=10):
    return np.convolve(values, np.ones(window) / window, mode="valid")


plt.figure(figsize=(8, 4))
for seed, (network, returns) in enumerate(runs):
    plt.plot(smooth(returns), label=f"seed {seed}")
plt.axhline(500, color="gray", linestyle=":", label="perfect score")
plt.title("DQN on CartPole: episode length during training")
plt.xlabel("episode"); plt.ylabel("steps the pole stayed up (smoothed)"); plt.grid(True); plt.legend()
plt.show()

for seed, (network, returns) in enumerate(runs):
    print(f"seed {seed}: last 10 training episodes {returns[-10:].mean():4.0f},  "
          f"final policy evaluated: {evaluate(env, network):4.0f}")

Put these numbers next to the ones from Section 5 of Notebook 08. There, the three runs ended far apart, and a good training score said little about the quality of the final policy. Here the three seeds behave alike, and for each of them the score at the end of training and the evaluation of the final policy agree.

DQN has not reached the perfect score of 500 in 20,000 steps, and its curves are not perfectly smooth either. What it has gained is **reliability**: the same procedure gives a comparable result each time. That is the property everything else is built on.

### Are both cures needed?

An **ablation** removes one component of a system to see what it contributes. Removing the target network is easy: with `tau=1.0` the target network is an exact copy of the online network after every step.

In [ ]:
print("DQN without a target network (tau = 1):")
for seed in range(3):
    network, returns = dqn(env, 20000, seed=seed, tau=1.0, **CARTPOLE)
    print(f"  seed {seed}: last 10 training episodes {returns[-10:].mean():4.0f},  "
          f"final policy evaluated: {evaluate(env, network):4.0f}")

Without the target network the agent learns nothing useful at all: the pole falls as fast as it can. The moving-target problem, which was a source of noise in Notebook 08, becomes fatal once the network is trained hard on minibatches. Exercise 3 removes the replay buffer and leaves the target network in.

## 7. Lunar Lander

### The task
CartPole is a toy. **Lunar Lander** is a small but real control problem. A spacecraft starts at the top of the screen with a random push and has to come down gently on the landing pad between two flags.

| | |
|---|---|
| **State** | 8 numbers: position (x, y), velocity (x, y), angle, turning speed, and whether each of the two legs touches the ground |
| **Actions** | 0 do nothing, 1 fire the left engine, 2 fire the main engine, 3 fire the right engine |
| **Reward** | moving towards the pad and slowing down is rewarded, moving away is penalised; firing engines costs a little; crashing gives −100 and landing +100 |
| **Solved** | an average return of **200** or more |

An episode ends when the lander crashes, comes to rest, flies off the screen, or after 1,000 steps.

In [ ]:
lander = gym.make("LunarLander-v3", render_mode="rgb_array")
print("observation space:", lander.observation_space.shape, "  actions:", lander.action_space.n)


def record(env, network=None, seed=0):
    """One episode as a list of frames. Without a network the actions are random."""
    state, info = env.reset(seed=seed)
    frames, total = [env.render()], 0.0
    terminated = truncated = False
    while not (terminated or truncated):
        if network is None:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = int(network(torch.as_tensor(state, dtype=torch.float32)).argmax())
        state, reward, terminated, truncated, info = env.step(action)
        total += reward
        frames.append(env.render())
    return frames, total


frames, total = record(lander, seed=1)
print(f"random agent: return {total:.0f}")
show_video(frames[::2], fps=25)

The random agent fires its engines without purpose and comes down wherever chance takes it, usually in a crash and with a negative return.

### Training
The network is larger than for CartPole (two hidden layers of 256 units), the minibatch is larger, and the other settings are those commonly used for this environment. Training stops as soon as the average return of the last 100 episodes reaches 200, or after 300,000 steps.

**This cell takes several minutes.** It prints a line every 50 episodes so that you can watch the progress: the average return starts around −150 and should climb past 0 and towards 200, though not steadily.

In [ ]:
LANDER = dict(hidden=256, learning_rate=6.3e-4, batch_size=128, buffer_size=50000, tau=0.01,
              explore_steps=36000, epsilon_end=0.1)

lander_network, lander_returns = dqn(lander, 300000, target_score=200, log=50, **LANDER)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(lander_returns, alpha=0.3, label="each episode")
plt.plot(np.arange(49, len(lander_returns)), smooth(lander_returns, 50), label="average of 50")
plt.axhline(200, color="green", linestyle="--", label="solved")
plt.title("DQN on Lunar Lander")
plt.xlabel("episode"); plt.ylabel("return"); plt.grid(True); plt.legend()
plt.show()

score = evaluate(gym.make("LunarLander-v3"), lander_network)
print(f"trained for {len(lander_returns)} episodes")
print(f"greedy policy, average return over 30 new episodes: {score:.0f}")

The faint line shows how much single episodes differ: even late in training some end badly. The average tells the story. At first the agent mostly crashes. Then it learns to slow its fall. There is often a phase in which it hovers without landing until the time runs out, which costs fuel and shows as a dip in the curve. Finally it learns to steer to the pad and set down.

In [ ]:
frames, total = record(lander, lander_network, seed=0)
print(f"trained agent: return {total:.0f}")
show_video(frames[::2], fps=25)

Change the seed to watch other descents. Nobody wrote a rule about engines, angles or speeds. The agent was given eight numbers, four buttons and a reward.

## 8. Keeping the result

Training took minutes and a Colab session forgets everything when it ends, so a trained network should be saved. PyTorch stores the weights in a file, and loading them into a fresh network of the same shape restores the agent.

In [ ]:
torch.save(lander_network.state_dict(), "lander_dqn.pt")

restored = make_network(8, 4, hidden=256)
restored.load_state_dict(torch.load("lander_dqn.pt"))

state, info = lander.reset(seed=5)
state = torch.as_tensor(state, dtype=torch.float32)
print("original:", lander_network(state).detach().numpy().round(2))
print("restored:", restored(state).detach().numpy().round(2))

In Colab the file `lander_dqn.pt` appears in the file panel on the left, from where it can be downloaded. Like everything else in a Colab session, it disappears when the session ends.

> ⚠️ **Common pitfalls**
>
> - **Computing the target with the online network.** The whole point of the second network is lost. The ablation in Section 6 shows the result.
> - **Storing `terminated or truncated` in the buffer.** Only `terminated` switches off the future term. An episode cut off by the time limit still had a future.
> - **Training before the buffer has warmed up.** The first few hundred transitions are too alike to learn from.
> - **Using settings from one environment in another.** CartPole and Lunar Lander needed different learning rates, network sizes and exploration schedules. Reported hyperparameters are a starting point.
> - **Expecting a monotonic curve.** Even a working DQN has bad episodes and temporary drops. Judge by the average and by a separate evaluation.

## 9. What DQN does not solve

- **The values are too optimistic.** The `max` in the target picks whichever action looks best, including actions that only look good because of an estimation error. Notebook 10 measures this and fixes it.
- **All transitions are replayed equally often**, although some are far more instructive than others. Notebook 10 again.
- **The actions must be few and discrete**, because the network needs one output per action. A robot joint that can turn by any angle does not fit. Module D takes another route.

---
## ✏️ Exercises

The exercises use `ReplayBuffer`, `dqn`, `evaluate`, `record`, `smooth`, `CARTPOLE`, `env`, and the trained `lander_network`.

### Exercise 1 (A buffer of three)
Create a `ReplayBuffer` with `capacity=3` and `state_size=1`. Add five transitions whose state is `[0.0]`, `[1.0]`, ... `[4.0]` (use 0 for the action and the reward, the same state as next state, and `False` for terminated). Predict which states are in the buffer afterwards and in which positions, then print `buffer.states`.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
small = ReplayBuffer(capacity=3, state_size=1)
for i in range(5):
    small.add([float(i)], 0, 0.0, [float(i)], False)
print(small.states.ravel())
print("size:", small.size, " next position to overwrite:", small.position)
```

*The buffer is a ring. Transitions 0, 1 and 2 fill it, then 3 overwrites position 0 and 4 overwrites position 1. The oldest surviving transition is 2.*
</details>

### Exercise 2 (How fast should the target follow?)
Section 6 showed that `tau = 1` fails. Train on CartPole (20,000 steps, seeds 0 to 2, the other settings as in `CARTPOLE`) with `tau = 0.0005` and with `tau = 0.05`, and compare with the notebook's `tau = 0.005`. Explain both failures.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for tau in (0.0005, 0.05):
    for seed in range(3):
        network, returns = dqn(env, 20000, seed=seed, tau=tau, **CARTPOLE)
        print(f"tau = {tau:<7} seed {seed}: last 10 training episodes {returns[-10:].mean():4.0f},  "
              f"final policy evaluated: {evaluate(env, network):4.0f}")
```

*With a very small tau the target network is held almost still. That is stable, but the targets are computed by a network that is nearly as ignorant as at the start, so in 20,000 steps little is learned. With a tau that is too large the target network follows the online network almost immediately, and we are back at the moving target of Section 6. The useful range is narrow, and it is one more setting that has to be found by experiment.*
</details>

### Exercise 3 (Without replay)
Keep the target network and remove the replay buffer. With `buffer_size=64, warm_up=64` the buffer only ever holds the 64 most recent transitions, so every minibatch is drawn from the last moments of play. Train on CartPole for seeds 0 to 2 and compare with the full DQN of Section 6.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
print("DQN with a buffer of only the last 64 transitions:")
for seed in range(3):
    network, returns = dqn(env, 20000, seed=seed, buffer_size=64, warm_up=64, **CARTPOLE)
    print(f"  seed {seed}: last 10 training episodes {returns[-10:].mean():4.0f},  "
          f"final policy evaluated: {evaluate(env, network):4.0f}")
```

*The agent still learns something, but the three seeds end far apart, and the training score and the evaluation no longer agree. That is the signature of Notebook 08: every update is fitted to what the agent is doing at this moment. The target network keeps the values from collapsing. The replay buffer is what makes the result repeatable.*
</details>

### Exercise 4 (How good is the lander really?)
An average hides a lot. Play 50 episodes with `lander_network` (seeds 0 to 49, a fresh environment without rendering is faster), collect the returns, and plot a histogram. How many episodes scored 200 or more? What is the worst one? Record a video of the worst episode.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
test_env = gym.make("LunarLander-v3")
returns = []
for seed in range(50):
    state, info = test_env.reset(seed=seed)
    total, terminated, truncated = 0.0, False, False
    while not (terminated or truncated):
        with torch.no_grad():
            action = int(lander_network(torch.as_tensor(state, dtype=torch.float32)).argmax())
        state, reward, terminated, truncated, info = test_env.step(action)
        total += reward
    returns.append(total)

returns = np.array(returns)
plt.figure(figsize=(7, 4))
plt.hist(returns, bins=20)
plt.axvline(200, color="green", linestyle="--")
plt.title("Returns of the trained lander over 50 episodes")
plt.xlabel("return"); plt.ylabel("episodes"); plt.grid(True)
plt.show()

print(f"average {returns.mean():.0f},  episodes with 200 or more: {(returns >= 200).sum()} of 50,  "
      f"worst: {returns.min():.0f} (seed {returns.argmin()})")
frames, total = record(lander, lander_network, seed=int(returns.argmin()))
show_video(frames[::2], fps=25)
```

*Most episodes are landings worth more than 200, and a few fall clearly short. Watch the weakest one and work out what went wrong in it. An agent that is "solved" on average can still do poorly in individual cases, and for a real spacecraft the worst case is the one that counts.*
</details>

### Exercise 5 (Wind (a little harder))
The lander was trained in still air. Gymnasium can add wind: `gym.make("LunarLander-v3", enable_wind=True, wind_power=10.0)`. Without any further training, evaluate `lander_network` in still air and with a wind power of 5, 10 and 15 (30 episodes each). Then record a video at wind power 15. What does this tell you about a trained agent and the conditions it was trained in?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
print(f"still air:     average return {evaluate(gym.make('LunarLander-v3'), lander_network):6.0f}")
for power in (5.0, 10.0, 15.0):
    windy = gym.make("LunarLander-v3", enable_wind=True, wind_power=power)
    print(f"wind power {power:>4}: average return {evaluate(windy, lander_network):6.0f}")

windy = gym.make("LunarLander-v3", enable_wind=True, wind_power=15.0, render_mode="rgb_array")
frames, total = record(windy, lander_network, seed=2)
print(f"video: return {total:.0f}")
show_video(frames[::2], fps=25)
```

*The return drops as soon as there is any wind. The agent has only ever seen states that occur in still air. Wind pushes it into states near those, and the network generalises to them well enough to land most of the time, but no longer at the level it reached in training. A learned policy can be relied on only for conditions like those it was trained in. If wind is to be expected, the agent has to be trained with wind.*
</details>

## 🔑 Key takeaways

- **Experience replay** stores transitions and trains on random minibatches: the samples are mixed, and each one is reused many times.
- Replay requires an **off-policy** algorithm, because the stored data come from older policies.
- The **target network** is a slowly updated copy used only to compute targets, so that the online network aims at something steady.
- **DQN** = Q-learning + neural network + experience replay + target network. Without the target network it failed completely on CartPole.
- The same code, with different settings, learned to land a spacecraft from 8 numbers and a reward.
- DQN is limited to **discrete actions** and tends to **overestimate** values, which leads to the next notebook.

---
**Next:** *Notebook 10: Better Deep Q-Networks*, where three refinements are put to work on Flappy Bird.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*